In [60]:
platformID = 'YT-'

## import libraries

In [61]:
from IPython.display import display

import os
import zipfile

from tqdm import tqdm 
from datetime import datetime

import pandas as pd
pd.set_option('display.max_colwidth', None)

import numpy as np

import re

#import yxdb

import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns 

import psycopg2

## import helper 

In [62]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config_GAM2025 import gam_info

from functions import execute_sql_query
import test_functions

In [63]:
gam_info['lookup_file']

'GAM2025_Lookup.xlsx'

In [64]:
# country
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID')

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])
week_tester['week_ending'] = pd.to_datetime(week_tester['week_ending'])

# social media accounts
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new')

socialmedia_accounts = socialmedia_accounts[(socialmedia_accounts['PlatformID'] == platformID)
                                            & 
                                            (socialmedia_accounts['Status'] == 'active')]
socialmedia_accounts = socialmedia_accounts.rename(columns={'Excluding UK': 'Channel Group'})

channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)
socialmedia_accounts.sample()

,PlatformID,Status,Comment,Channel ID,Channel Name,Service,ServiceID,Channel Group,Channel URL,Channel Username,Linked FB Account,Year
543,YT-,active,NaN,UCvm8V4Dwtz0fbPGbpk7RYAg,BBC News اردو,Urdu,URD,BBC World Service,NaN,NaN,NaN,GAM2025


# Unique Viewers

In [65]:
uniqueViewer_df = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_uniqueViewer.csv")
uniqueViewer_df.sample()


,Channel ID,Channel Name,ServiceID,Channel Group,Channel title,Unique viewers,w/c
2821,UCcOc3TjyW-utzRMOB7CXDEA,Bluey - Deutsch Offizieller Kanal,WOR,BBCW Family,Bluey - Deutsch Offizieller Kanal,519422.374186,2024-10-14


In [66]:
uniqueViewer_df[(uniqueViewer_df['ServiceID'] == 'SER') & 
                (uniqueViewer_df['w/c'] == '2024-04-01')]

,Channel ID,Channel Name,ServiceID,Channel Group,Channel title,Unique viewers,w/c
1763,UCCrAKchnDFMhrKeXSYWXjGA,BBC News na srpskom,SER,BBC World Service,BBC News na srpskom,19566.0,2024-04-01


# Country

In [67]:
country_df = pd.read_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_country_new.csv")
country_df.sample()


,w/c,Channel ID,PlaceID,country_%
344018,2024-09-30,UCN5piaaZEZBfvFJLd_kBHnA,GAM,0.000003


In [68]:
temp = country_df.merge(socialmedia_accounts[['Channel ID', 'Service']], on='Channel ID', how='left')
temp[(temp['PlaceID'] == 'INO')  & (temp['w/c'] == '2024-07-01') & (temp['Service'] == 'Media Action')]

,w/c,Channel ID,PlaceID,country_%,Service
936442,2024-07-01,aksikitaindo,INO,0.98267,Media Action
936503,2024-07-01,Sajha Sawal,INO,0.00000,Media Action


# combine UV & country

In [69]:
yt_uv_country = uniqueViewer_df.merge(country_df, 
                            on=['Channel ID', 'w/c'], 
                            how = 'outer', indicator=True)

################################### Testing ################################### 
test_step = 'merging uv & country'

test_functions.test_inner_join(uniqueViewer_df, country_df, ['Channel ID', 'w/c'], '1_YT_18', test_step)

################################### Testing ################################### 

print(yt_uv_country._merge.value_counts())

Inner join test 1_YT_18 failed: Issues found.
Issues with df_left (rows present in df_left but not in df_right)
Issues with df_right (rows present in df_right but not in df_left)
...updating logbook...

_merge
both          942238
right_only      2566
left_only       1529
Name: count, dtype: int64


In [70]:
#yt_uv_country.to_csv('temp_yt_uvCountry.csv', index=None)

In [71]:
yt_uv_country = yt_uv_country[yt_uv_country['_merge'] == 'both'].drop(columns=['_merge'])
yt_uv_country['uv_by_country'] = yt_uv_country['country_%'] * yt_uv_country['Unique viewers']
yt_uv_country.sample()

,Channel ID,Channel Name,ServiceID,Channel Group,Channel title,Unique viewers,w/c,PlaceID,country_%,uv_by_country
877503,UCvm8V4Dwtz0fbPGbpk7RYAg,BBC News اردو,URD,BBC World Service,BBC News اردو,1437322.0,2024-10-07,DJI,0.000014,20.649607


In [72]:
################################### Testing ################################### 
# all weeks 
# all weeks per channel
# all weeks per service
# all weeks per country

# duplicates

# test unique viewer is larger than unique vieewr per country 
# test total of unique vieewr per country == unique viewer
# test country sum == 1

# test allowed values - placeID
# test allowed values - ServiceID

################################### Testing ################################### 

country tests
- [ ] check only one entry per country & week & channel
- [ ] check that sum of unique views per country == unique views gathered from yt clickedicklick
- [ ] check country sum==1 (groupby channel & week == 1)
- [ ] county the number of weeks we have for every channel counntry combination
- [ ] check that no channel name is empty

In [73]:
cols = ['w/c', 'PlaceID', 'ServiceID', 'Channel ID', 'uv_by_country', ]
yt_uv_country[cols].to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_uniqueViewer_country.csv", 
                     index=None)
'''yt_uv_country.to_csv(f"../data/singlePlatform/input/YouTube/{gam_info['file_timeinfo']}_metric_country.csv", 
                     index=None)'''


'yt_uv_country.to_csv(f"../data/singlePlatform/input/YouTube/{gam_info[\'file_timeinfo\']}_metric_country.csv", \n                     index=None)'